# NRAM (Noise-Robust Artifact Modulation) Example

## Overview

**NRAM**은 딥페이크 탐지를 위한 Test-Time Adaptation 방법입니다.

**핵심 특징:**
- ✅ **간결한 코드**: 531줄 (기존 1028줄 대비 48% 감소)
- ✅ **3가지 클래스**: NRAMConfig, NRAM, UnifiedNRAM
- ✅ **Data-Driven**: Robust normalization (median/MAD)
- ✅ **Top-ρ Sample Selection**: Evidence 기반
- ✅ **Artifact-Conditional Attention**: Dual-path attention
- ✅ **Noise Gating**: 노이즈에 강한 적응

**Architecture:**
```
Base Model (frozen) → layer4 features
    ↓
NRAM Module
    ├─ Artifact Detector (FFT-based)
    ├─ Noise Estimator (Laplacian)
    ├─ Artifact-Conditional Attention
    └─ Noise Gating + Residual
    ↓
Enhanced Features → Classifier
    ↓
Final Prediction
```

## 1. Import

In [ ]:
import sys
# Clear cache
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['NPR', 'npr', 'LGrad', 'lgrad', 'nram']):
        del sys.modules[mod]

In [ ]:
import os
from pathlib import Path
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

# Dataset and metrics
from utils.data.dataset import CorruptedDataset
from utils.eval.metrics import PredictionCollector, MetricsCalculator

# Models
from model.LGrad.lgrad_model import LGrad
from model.NPR.npr_model import NPR

# NRAM (New simplified version)
from model.method.nram import UnifiedNRAM, NRAMConfig

In [ ]:
!nvidia-smi

## 2. Configuration

In [ ]:
# Device
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Model selection
MODEL = "NPR"  # "LGrad" or "NPR"

# Datasets - using new corrupted_dataset structure
DATASETS = ["ADM", "DDPM", "IDDPM", "LDM", "PNDM", "VQDIFFUSION", "SDV1", "SDV2", "PROGAN", "STYLEGAN", "STYLEGAN2", "BIGGAN", "CYCLEGAN", "STARGAN", "GAUGAN", "DEEPFAKE"]  # Can add more: "BIGGAN", "CYCLEGAN", "DDPM", etc.

# Corruptions and Severities (OST-style order)
# Order: color_contrast -> color_saturation -> resize -> gaussian_blur
# Each corruption: severity 1 -> 2 -> 3 -> 4 -> 5
CORRUPTIONS = ["color_contrast", "color_saturation", "resize", "gaussian_blur"]
SEVERITIES = ["corrupted1", "corrupted2", "corrupted3", "corrupted4", "corrupted5"]

# Paths
DATA_ROOT = "corrupted_dataset"

# NRAM config
REDUCTION_RATIO = 16
MAX_TTA_STEPS = 10
TTA_LR = 1e-5
UPDATE_SAMPLE_RATIO = 0.7
ENABLE_TTA = True  # Set to False for no TTA

BATCH_SIZE = 16

## 3. Load Base Model

In [ ]:
if MODEL == "LGrad":
    STYLEGAN_WEIGHTS = "model/LGrad/weights/karras2019stylegan-bedrooms-256x256_discriminator.pth"
    CLASSIFIER_WEIGHTS = "model/LGrad/weights/LGrad-Pretrained-Model/LGrad-4class-Trainon-Progan_car_cat_chair_horse.pth"
    
    base_model = LGrad(
        stylegan_weights=STYLEGAN_WEIGHTS,
        classifier_weights=CLASSIFIER_WEIGHTS,
        device=DEVICE
    )
    
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
    ])
    
elif MODEL == "NPR":
    NPR_WEIGHTS = "model/NPR/weights/NPR.pth"
    
    base_model = NPR(
        weights=NPR_WEIGHTS,
        device=DEVICE
    )
    
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

print(f"✅ {MODEL} base model loaded")

## 4. Create NRAM Model

In [ ]:
# Create NRAM config
config = NRAMConfig(
    model=MODEL,
    target_layer=None,  # Auto-detect
    reduction_ratio=REDUCTION_RATIO,
    max_tta_steps=MAX_TTA_STEPS,
    tta_lr=TTA_LR,
    update_sample_ratio=UPDATE_SAMPLE_RATIO,
    enable_tta=ENABLE_TTA,
    device=DEVICE,
)

# Create NRAM model
nram_model = UnifiedNRAM(base_model, config)

# Print info
print("="*60)
print("NRAM Model Configuration")
print("="*60)
print(f"Model: {config.model}")
print(f"Target Layer: {config.target_layer}")
print(f"Reduction Ratio: {config.reduction_ratio}")
print(f"Max TTA Steps: {config.max_tta_steps}")
print(f"TTA LR: {config.tta_lr}")
print(f"Update Sample Ratio (ρ): {config.update_sample_ratio}")
print(f"Enable TTA: {config.enable_tta}")
print("="*60)

## 5. Model Ready

**No training needed!** The model is ready to use.

In [ ]:
print("✅ NRAM model ready for inference")
if config.enable_tta:
    print("   TTA enabled: Model will adapt at test-time")
    print(f"   - Top-{config.update_sample_ratio*100:.0f}% sample selection")
    print(f"   - Max {config.max_tta_steps} adaptation steps")
else:
    print("   TTA disabled: Using base model + NRAM without adaptation")

## 6. Create Dataset

In [ ]:
dataset = CorruptedDataset(
    root=DATA_ROOT,
    datasets=DATASETS,
    corruptions=CORRUPTIONS,
    severities=SEVERITIES,
    transform=transform
)

print(f"Total samples: {len(dataset)}")
print(f"Datasets: {DATASETS}")
print(f"Corruptions: {CORRUPTIONS}")
print(f"Severities: {SEVERITIES}")
print(f"\nExperiment order (Gradual TTA style):")
for corruption in CORRUPTIONS:
    print(f"  {corruption}: severity 1 -> 2 -> 3 -> 4 -> 5")

## 7. Evaluation Functions

In [ ]:
def evaluate_nram(model, dataloader, device, name="test"):
    """
    Evaluate NRAM model on entire dataloader.
    
    TTA happens automatically inside model.forward() if enable_tta=True.
    """
    model.eval()
    collector = PredictionCollector()
    calc = MetricsCalculator()
    
    pbar = tqdm(dataloader, desc=name)
    for batch in pbar:
        images, labels, metadata = batch
        images = images.to(device)
        
        # Forward (TTA happens automatically if enabled)
        logits = model(images)
        probs = torch.sigmoid(logits).squeeze(1)
        
        # Collect predictions
        collector.update(labels, probs.cpu(), threshold=0.5)
    
    # Compute metrics
    metrics = calc.compute_from_collector(collector, name=name)
    return metrics

In [ ]:
def evaluate_base_model(model, dataloader, device, name="base"):
    """
    Evaluate base model without TTA.
    """
    model.eval()
    collector = PredictionCollector()
    calc = MetricsCalculator()

    pbar = tqdm(dataloader, desc=name)
    with torch.no_grad():
        for batch in pbar:
            images, labels, metadata = batch
            images = images.to(device)

            logits = model(images)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]
            probs = torch.sigmoid(logits).squeeze(1)
            collector.update(labels, probs.cpu(), threshold=0.5)

    metrics = calc.compute_from_collector(collector, name=name)
    return metrics

## 8. Full Evaluation

In [ ]:
# Evaluate on all combinations
# Order: corruption -> severity (Gradual TTA style)
# e.g., color_contrast/1 -> color_contrast/2 -> ... -> color_contrast/5 -> color_saturation/1 -> ...

calc = MetricsCalculator()
all_results = {}  # {(dataset, corruption, severity): metrics}
corruption_avg_results = {}  # {(dataset, corruption): averaged_metrics}

for dataset_name in DATASETS:
    # Reset NRAM model when switching to a new dataset
    nram_model.reset()
    print(f"\n{'#'*70}")
    print(f"# Dataset: {dataset_name} (NRAM model reset)")
    print(f"{'#'*70}")
    
    for corruption in CORRUPTIONS:
        severity_results = []  # Store results for severity 1-5
        
        print(f"\n{'='*70}")
        print(f"Corruption: {corruption}")
        print(f"{'='*70}")
        
        for severity in SEVERITIES:
            # Get indices for this specific combination
            indices = [
                i for i, s in enumerate(dataset.samples)
                if s['dataset'] == dataset_name 
                and s['corruption'] == corruption 
                and s['severity'] == severity
            ]
            
            if len(indices) == 0:
                print(f"  {severity}: No samples, skipping")
                continue
            
            print(f"\n  [{severity}] Samples: {len(indices)}")
            
            # Create dataloader (shuffle=False to maintain order)
            dataloader = DataLoader(
                Subset(dataset, indices),
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=4,
                drop_last=True
            )
            
            # Evaluate with NRAM (TTA happens automatically if enabled)
            # Note: Model state is maintained across severities (continual adaptation)
            metrics = evaluate_nram(
                model=nram_model,
                dataloader=dataloader,
                device=DEVICE,
                name=f"{corruption}-{severity}"
            )
            
            # Print results for this severity
            print(f"      Acc: {metrics['accuracy']*100:.2f}%  AUC: {metrics['auc']*100:.2f}%  AP: {metrics['ap']*100:.2f}%  F1: {metrics['f1']*100:.2f}%")
            
            # Store results
            all_results[(dataset_name, corruption, severity)] = metrics
            severity_results.append(metrics)
        
        # Compute average across severities 1-5 for this corruption
        if severity_results:
            avg_metrics = {
                'accuracy': np.mean([m['accuracy'] for m in severity_results]),
                'auc': np.mean([m['auc'] for m in severity_results]),
                'ap': np.mean([m['ap'] for m in severity_results]),
                'f1': np.mean([m['f1'] for m in severity_results]),
            }
            corruption_avg_results[(dataset_name, corruption)] = avg_metrics
            
            print(f"\n  >> {corruption} Average (severity 1-5):")
            print(f"      Acc: {avg_metrics['accuracy']*100:.2f}%  AUC: {avg_metrics['auc']*100:.2f}%  AP: {avg_metrics['ap']*100:.2f}%  F1: {avg_metrics['f1']*100:.2f}%")

# ============================================================
# Summary Tables
# ============================================================
print(f"\n\n{'='*80}")
print(f"RESULTS SUMMARY (NRAM - TTA {'Enabled' if ENABLE_TTA else 'Disabled'})")
print(f"{'='*80}")

# Table 1: Detailed results per severity
print(f"\n[Table 1] Detailed Results (per severity)")
print(f"{'Corruption':<20} {'Severity':<12} {'Accuracy':<10} {'AUC':<10} {'AP':<10} {'F1':<10}")
print("-" * 72)
for (dataset_name, corruption, severity), metrics in all_results.items():
    print(f"{corruption:<20} {severity:<12} {metrics['accuracy']*100:>6.2f}%    {metrics['auc']*100:>6.2f}%    {metrics['ap']*100:>6.2f}%    {metrics['f1']*100:>6.2f}%")

# Table 2: Averaged results per corruption (main result)
print(f"\n\n[Table 2] Averaged Results (severity 1-5 mean) - MAIN RESULT")
print(f"{'Corruption':<25} {'Accuracy':<10} {'AUC':<10} {'AP':<10} {'F1':<10}")
print("-" * 65)
for (dataset_name, corruption), metrics in corruption_avg_results.items():
    print(f"{corruption:<25} {metrics['accuracy']*100:>6.2f}%    {metrics['auc']*100:>6.2f}%    {metrics['ap']*100:>6.2f}%    {metrics['f1']*100:>6.2f}%")

# Overall average
overall_avg = {
    'accuracy': np.mean([m['accuracy'] for m in corruption_avg_results.values()]),
    'auc': np.mean([m['auc'] for m in corruption_avg_results.values()]),
    'ap': np.mean([m['ap'] for m in corruption_avg_results.values()]),
    'f1': np.mean([m['f1'] for m in corruption_avg_results.values()]),
}
print("-" * 65)
print(f"{'Overall Average':<25} {overall_avg['accuracy']*100:>6.2f}%    {overall_avg['auc']*100:>6.2f}%    {overall_avg['ap']*100:>6.2f}%    {overall_avg['f1']*100:>6.2f}%")

## Summary

### NRAM Key Features:

#### 1. **간결한 구조**
- 3가지 클래스만 사용: `NRAMConfig`, `NRAM`, `UnifiedNRAM`
- 531줄 (기존 1028줄 대비 48% 감소)
- NORM.py와 동일한 스타일

#### 2. **Data-Driven Normalization**
- Robust normalization (median/MAD)
- Quantile-based sample selection
- No magic numbers!

#### 3. **핵심 컴포넌트**
- Frequency Artifact Detector (FFT-based)
- Noise Estimator (Laplacian filter)
- Artifact-Conditional Attention (Dual-path)
- Noise Gating + Residual Connection

#### 4. **Test-Time Adaptation**
- Evidence-based sample selection (top-ρ)
- Entropy minimization on selected samples
- Automatic parameter adaptation

#### 5. **간단한 사용법**
```python
# Simple usage
from model.method.nram import UnifiedNRAM, NRAMConfig

config = NRAMConfig(model="NPR", enable_tta=True)
model = UnifiedNRAM(base_model, config)

logits = model(images)  # TTA happens automatically!
probs = torch.sigmoid(logits)
```